# QC & Filtering

Quality control and cell filtering for scRNA-seq data (nf-core/scrnaseq h5ad output).

This notebook:
1. Loads an h5ad file produced by nf-core/scrnaseq
2. Computes QC metrics (mitochondrial %, gene counts, UMI counts)
3. Generates diagnostic plots (violin, scatter)
4. Applies filtering thresholds
5. Saves the filtered h5ad for downstream analysis

In [ ]:
# Parameters (injected by Papermill)
input_h5ad_path: str = "/data/samples/sample.h5ad"
experiment_name: str = "experiment"
mito_threshold: float = 20.0
min_genes: int = 200
min_cells: int = 3
bioaf_api_url: str = "http://localhost:8000"
experiment_id: str = ""

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False, figsize=(6, 4))

results_dir = f"/data/results/{experiment_name}"
figures_dir = os.path.join(results_dir, "figures", "qc")
os.makedirs(figures_dir, exist_ok=True)

print(f"Input: {input_h5ad_path}")
print(f"Results dir: {results_dir}")

## 1. Load Data

In [ ]:
adata = sc.read_h5ad(input_h5ad_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")
adata

## 2. Calculate QC Metrics

Identify mitochondrial genes and compute per-cell QC statistics.

In [ ]:
# Identify mitochondrial genes (human MT- or mouse mt-)
adata.var["mt"] = adata.var_names.str.startswith(("MT-", "mt-"))
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

print(f"Mitochondrial genes found: {adata.var['mt'].sum()}")
print(f"Median genes/cell: {np.median(adata.obs['n_genes_by_counts']):.0f}")
print(f"Median UMIs/cell: {np.median(adata.obs['total_counts']):.0f}")
print(f"Median mito %: {np.median(adata.obs['pct_counts_mt']):.1f}%")

## 3. QC Violin Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sc.pl.violin(adata, "n_genes_by_counts", ax=axes[0], show=False)
axes[0].set_title("Genes per Cell")
axes[0].axhline(y=min_genes, color="r", linestyle="--", alpha=0.7)

sc.pl.violin(adata, "total_counts", ax=axes[1], show=False)
axes[1].set_title("UMI Counts per Cell")

sc.pl.violin(adata, "pct_counts_mt", ax=axes[2], show=False)
axes[2].set_title("Mitochondrial %")
axes[2].axhline(y=mito_threshold, color="r", linestyle="--", alpha=0.7)

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, "qc_violin.png"), bbox_inches="tight")
plt.show()

## 4. QC Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"], s=1, alpha=0.3)
axes[0].set_xlabel("UMI Counts")
axes[0].set_ylabel("Genes Detected")
axes[0].set_title("Genes vs UMIs")
axes[0].axhline(y=min_genes, color="r", linestyle="--", alpha=0.7)

axes[1].scatter(adata.obs["total_counts"], adata.obs["pct_counts_mt"], s=1, alpha=0.3)
axes[1].set_xlabel("UMI Counts")
axes[1].set_ylabel("Mitochondrial %")
axes[1].set_title("Mito % vs UMIs")
axes[1].axhline(y=mito_threshold, color="r", linestyle="--", alpha=0.7)

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, "qc_scatter.png"), bbox_inches="tight")
plt.show()

## 5. Apply Filters

In [ ]:
n_before = adata.n_obs

sc.pp.filter_cells(adata, min_genes=min_genes)
sc.pp.filter_genes(adata, min_cells=min_cells)
adata = adata[adata.obs["pct_counts_mt"] < mito_threshold, :].copy()

n_after = adata.n_obs
print(f"Cells before filtering: {n_before}")
print(f"Cells after filtering:  {n_after}")
print(f"Cells removed: {n_before - n_after} ({(n_before - n_after) / n_before * 100:.1f}%)")
print(f"Genes retained: {adata.n_vars}")

## 6. Post-filter QC

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sc.pl.violin(adata, "n_genes_by_counts", ax=axes[0], show=False)
axes[0].set_title("Genes per Cell (filtered)")
sc.pl.violin(adata, "total_counts", ax=axes[1], show=False)
axes[1].set_title("UMI Counts (filtered)")
sc.pl.violin(adata, "pct_counts_mt", ax=axes[2], show=False)
axes[2].set_title("Mito % (filtered)")
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, "qc_violin_filtered.png"), bbox_inches="tight")
plt.show()

## 7. Save Filtered Data

In [ ]:
output_path = os.path.join(results_dir, "01_qc_filtered.h5ad")
adata.write_h5ad(output_path)
print(f"Saved filtered data to: {output_path}")
print(f"Final shape: {adata.n_obs} cells x {adata.n_vars} genes")